# LC 846 — Hand of Straights
**Difficulty:** Medium &nbsp;|&nbsp; **Category:** Greedy
**Pattern:** Sort + Ordered Map — Consume Groups Greedily

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Always start a new
group from the smallest remaining card. If any
card needed to complete the consecutive group of
size W is missing, it is impossible. Use an
ordered frequency map to consume cards in order.
</div>

## Official Problem Statement

Alice has some number of cards and she wants to
rearrange the cards into groups so that each
group is of size `groupSize`, and consists of
`groupSize` consecutive cards.

Given an integer array `hand` where `hand[i]` is
the value written on the `i`th card and an integer
`groupSize`, return `true` if she can rearrange
the cards, or `false` otherwise.

**Example 1:**
```
Input:  hand = [1,2,3,6,2,3,4,7,8], groupSize = 3
Output: true
Explanation: [1,2,3], [2,3,4], [6,7,8]
```
**Example 2:**
```
Input:  hand = [1,2,3,4,5], groupSize = 4
Output: false
```

**Constraints:**
- `1 <= hand.length <= 10^4`
- `0 <= hand[i] <= 10^9`
- `1 <= groupSize <= hand.length`

## What This Is Actually Asking

Split a deck of cards into equal groups of W
consecutive cards. For example W=3 means each
group looks like [4,5,6] or [8,9,10]. Return
True if all cards can be split this way, False
if any card is left over or any group is broken.

## Walk Through an Example by Hand

```
hand = [1,2,3,6,2,3,4,7,8]  groupSize=3

Frequency map (ordered): {1:1, 2:2, 3:2, 4:1, 6:1, 7:1, 8:1}

Smallest card = 1, count=1 -> need 1 group starting at 1
  Consume 1 group [1,2,3]:
    count[1] -= 1 -> 0  remove
    count[2] -= 1 -> 1
    count[3] -= 1 -> 1
  Map: {2:1, 3:1, 4:1, 6:1, 7:1, 8:1}

Smallest = 2, count=1 -> need 1 group starting at 2
  Consume [2,3,4]:
    count[2]-=1 -> 0  remove
    count[3]-=1 -> 0  remove
    count[4]-=1 -> 0  remove
  Map: {6:1, 7:1, 8:1}

Smallest = 6, count=1 -> need 1 group starting at 6
  Consume [6,7,8]: all removed
  Map: {}

Map empty -> return True
```

## The Picture

```
hand = [1,2,3,6,2,3,4,7,8]  W=3

Sort and view as groups:
  1  2  2  3  3  4  6  7  8
  |--group1--|  |--g2--|  |--g3--|
  [1, 2, 3]    [2, 3, 4]  [6, 7, 8]

Algorithm:
  freq = Counter(hand)   # {1:1, 2:2, 3:2, 4:1, 6:1...}
  sorted_keys = sorted(freq)  # process smallest first

  For smallest key k with count c:
    Need c groups all starting at k
    Consume k, k+1, ..., k+W-1 each by c:
      if any of those keys has count < c -> False
      else decrement each by c, remove zeros

Why smallest first?
  The smallest card MUST start a new group.
  It can never be the middle or end of a group
  (there is nothing smaller to start from).
```

## When To Use This Pattern

- When grouping consecutive values greedily,
  think **sort + ordered frequency map**
- When the smallest remaining value must start
  a group, think **greedy — no other choice**
- When `len(hand) % groupSize != 0`, think
  **return False immediately — can't divide evenly**
- When a required card is missing, think
  **return False — group cannot be completed**

## The Approach

If the hand size is not divisible by groupSize,
return False immediately. Build a frequency counter
and sort the unique values. For the smallest key,
try to form as many groups as its count allows:
subtract the count from each of the next W
consecutive values. If any are missing or
insufficient, return False. Return True when all
cards are consumed.

In [ ]:
from collections import Counter  # frequency map
from typing import List

In [ ]:
def test_harness(func):
    tests = [
        # (hand, groupSize, expected)
        ([1,2,3,6,2,3,4,7,8], 3, True),
        ([1,2,3,4,5],          4, False),
        ([1,2,3],              3, True),   # single group
        ([1,2,3,4],            4, True),
        ([1,2,3,4],            2, True),   # [1,2],[3,4]
        ([1,1,2,2,3,3],        3, True),   # two groups [1,2,3]
        ([1,2,3,3,4,5],        3, True),
        ([1,1,1],              3, False),  # not consecutive
        ([1],                  1, True),
    ]

    passed = 0
    for i, (hand, gs, expected) in enumerate(tests):
        result = func(hand[:], gs)
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"hand={hand} W={gs} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [ ]:
def isNStraightHand(
    hand: List[int], groupSize: int
) -> bool:
    """
    Return True if hand can be split into consecutive
    groups of size groupSize.

    If len(hand) % groupSize != 0: False. Build Counter
    and sort keys. For smallest key k with count c:
    subtract c from k..k+W-1; if any missing return False.

    Time:  O(n log n) — dominated by sort
    Space: O(n) — frequency map
    """
    pass


# Quick debug — run this cell while building
print(isNStraightHand([1,2,3,6,2,3,4,7,8], 3))  # True
print(isNStraightHand([1,2,3,4,5], 4))            # False
print(isNStraightHand([1,2,3], 3))                # True
print(isNStraightHand([1,1,1], 3))                # False

In [ ]:
# Uncomment and run when solution is ready
# test_harness(isNStraightHand)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force — try all permutations | O(n!) | O(n) |
| Sort + Counter greedy | O(n log n) | O(n) |

Sorting enables the greedy insight: the smallest
card must start a group. Once that decision is
locked in, consume the required consecutive cards.
No backtracking needed.

## Real World Connection

At Citi, trading settlement requires grouping
consecutive trade IDs into batches of fixed size W
for regulatory reporting. If any ID in a required
consecutive range is missing (failed trade), the
batch cannot be formed.
Hand of Straights models this exactly: the
greedy smallest-first approach catches the first
missing ID immediately and returns False before
processing the entire batch.
On AWS Kinesis, record sequence numbers must be
consumed in consecutive windows of fixed size —
the same algorithm validates completeness before
a batch is committed downstream.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra